# 作业：RAG 系统导论
---

欢迎来到 RAG 课程的第一份作业！在本作业中，你将使用包含 BBC 新闻信息的数据集构建一个 RAG 流程。目标是让 LLM 从数据集中检索相关新闻细节，并利用这些信息生成更有依据的回答。你将使用的模型是 [llama-3-1-8b-instruct-turbo](https://www.together.ai/models/llama-3-1)，其训练数据截至 2023 年 12 月。本作业的思路是构建一个 RAG 系统，使其能够纳入 2024 年发生事件的信息。

在本作业中，你将：
- 使用查询与检索函数，根据 query 获取相关数据
- 对检索到的数据进行适当格式化
- 将 query 与相关数据组合成提示词并输入 LLM

---
<h4 style="color:black; font-weight:bold;">使用目录</h4>

JupyterLab 提供了便捷方式帮助你浏览本作业。目录位于左侧面板的 Table of Contents 选项卡中，如下图所示。

![TOC Location](images/toc.png)

---

<h4 style="color:green; font-weight:bold;">作业顺利评分提示：</h4>

- 除需提交答案的单元（或明确说明可交互的单元）外，其余单元均已冻结。

- 你可以新增单元进行实验，但评分器会忽略这些新单元，因此不要把最终解答写在新建单元里，请使用指定位置。

- 除非绝对必要，尽量避免使用全局变量。评分器会在隔离环境中测试你的代码，不会从头依次运行全部单元，因此全局变量在评分时可能不可用。若某个全局变量需要使用，通常会以全大写形式定义。

- 提交作业前，请先点击页面左上角的 💾 图标保存 notebook，然后点击右上角的 <span style="background-color: gray; color: white; padding: 3px 5px; font-size: 16px; border-radius: 5px;">Submit assignment</span> 按钮。
---

<a id='1'></a>
## 1 - 介绍

---

### 1.1 RAG 架构概览

如课程中所示，简化版 RAG 示意图如下：

<div align="center">
  <img src="images/rag_overview.png" alt="RAG Overview" width="60%">
</div>

本作业将引导你完成整体工作流程。你将使用预先实现的检索器，为给定 query 获取相关数据，对数据进行格式化，并创建一个同时包含 query 与检索信息的新提示词。最后，你将比较使用和不使用 RAG 系统时的查询结果，以评估其对 LLM 回答的影响。

准备好了就开始吧！

### 1.2 导入所需库

运行下方单元以导入本作业所需的库。

In [1]:
from utils import (
    retrieve, 
    pprint, 
    generate_with_single_input, 
    read_dataframe, 
    display_widget
)
import unittests

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

<a id='2'></a>

## 2 - 加载数据集

---

在本作业中，你将使用 Kaggle 数据集 [News Headlines 2024](https://www.kaggle.com/datasets/dylanjcastillo/news-headlines-2024)。该数据集包含来自 BBC News 的数千条新闻标题及相关信息。


In [2]:
NEWS_DATA = read_dataframe("news_data_dedup.csv")

我们先看一下数据结构。

In [3]:
pprint(NEWS_DATA[9:11])

[
  {
    "guid": "5dae28f191cfd1047f67c409e616fc3f",
    "title": "Paris's Moulin Rouge loses windmill sails overnight",
    "description": "The cause of the sails' collapse from the roof of the world famous cabaret club is not yet clear.",
    "venue": "BBC",
    "url": "https://www.bbc.co.uk/news/world-europe-68895836",
    "published_at": "2024-04-25",
    "updated_at": "2024-04-26"
  },
  {
    "guid": "d2c3ff79d4e068911d05416ca061cd51",
    "title": "Ukraine uses longer-range US missiles for first time",
    "description": "Missiles secretly delivered this month have been used to strike Russian targets in Crimea, US media say.",
    "venue": "BBC",
    "url": "https://www.bbc.co.uk/news/world-europe-68893196",
    "published_at": "2024-04-25",
    "updated_at": "2024-04-26"
  }
]


关键字段包括 `title`、`description`、`url` 和 `published_at`。这些字段能为 LLM 提供足够的信息，以较好地回答大多数问题。

<a id='3'></a>
## 3 - 主要函数
---
将为你提供两个函数：
- `query_news`：给定一组索引，返回对应索引的全部文档。
- `retrieve`：给定 query 与整数 `top_k`，返回最相关的 `top_k` 篇文档。

你需要实现的函数有：
- `get_relevant_data`：接收 query 与 `top_k`，返回 `top_k` 个相关文档。
- `format_relevant_data`：给定文档列表，生成包含文档信息的格式化字符串。

随后你将基于这些函数构建自己的小型 RAG 流程，并观察其效果！

<a id='3-1'></a>
### 3.1 按索引查询新闻函数

这个简单函数用于根据索引列表返回文档，已为你提供。

In [ ]:
def query_news(indices):
    """
    根据给定索引，从数据集中取回对应元素。

    参数：
    indices (list of int)：包含目标元素索引的列表。
    
    返回：
    list：与给定索引对应的数据元素列表。
    """
     
    output = [NEWS_DATA[index] for index in indices]

    return output

In [ ]:
# 获取一些索引
indices = [3, 6, 9]
pprint(query_news(indices))

[
  {
    "guid": "e696224ac208878a5cec8bdc9f97c632",
    "title": "Europe risks dying and faces big decisions - Macron",
    "description": "The French president delivers a stark warning for Europe to act fast to survive in a changing world.",
    "venue": "BBC",
    "url": "https://www.bbc.co.uk/news/world-europe-68898887",
    "published_at": "2024-04-25",
    "updated_at": "2024-04-26"
  },
  {
    "guid": "4f585bad8f61b715fbafe2f022ab0ae8",
    "title": "Supreme Court divided on whether Trump has immunity",
    "description": "The justices discussed immunity, coups, pardons, Operation Mongoose - and the future of democracy.",
    "venue": "BBC",
    "url": "https://www.bbc.co.uk/news/world-us-canada-68901817",
    "published_at": "2024-04-25",
    "updated_at": "2024-04-26"
  },
  {
    "guid": "5dae28f191cfd1047f67c409e616fc3f",
    "title": "Paris's Moulin Rouge loses windmill sails overnight",
    "description": "The cause of the sails' collapse from the roof of the world famou

<a id='3-2'></a>
### 3.2 检索函数

`retrieve` 函数是本 RAG 系统的关键组件。完整代码已在 `utils.py` 中提供，你可以自行查看。本阶段重点是理解它的输入参数和输出结果。在第 2 模块中，你将学习其更详细的原理以及多种文档检索技术。

**参数：**
- `query`：字符串类型的搜索查询，用于查找最相关文档。
- `top_k`：整数，表示返回相似度最高的文档数量。

**输出：**
- 函数返回一个索引列表，对应语料库中与 query 最相似的前 `k` 篇文档（基于相似度得分）。

**调用方式：**

```Python
retrieve(query: str, top_k: int)
```

随着课程推进，你将更深入理解该函数如何利用嵌入与相似度度量来实现高效文档检索。

In [ ]:
# 测试 retrieve 函数！
indices = retrieve("Concerts in North America", top_k = 1)
print(indices)

[350]


In [ ]:
# 现在查询对应的新闻数据
retrieved_documents = query_news(indices)
pprint(retrieved_documents)

[
  {
    "guid": "927257674585bb6ef669cf2c2f409fa7",
    "title": "\u2018The working class can\u2019t afford it\u2019: the shocking truth about the money bands make on tour",
    "description": "As Taylor Swift tops $1bn in tour revenue, musicians playing smaller venues are facing pitiful fees and frequent losses. Should the state step in to save our live music scene?When you see a band playing to thousands of fans in a sun-drenched festival field, signing a record deal with a major label or playing endlessly from the airwaves, it\u2019s easy to conjure an image of success that comes with some serious cash to boot \u2013 particularly when Taylor Swift has broken $1bn in revenue for her current Eras tour. But looks can be deceiving. \u201cI don\u2019t blame the public for seeing a band playing to 2,000 people and thinking they\u2019re minted,\u201d says artist manager Dan Potts. \u201cBut the reality is quite different.\u201dPost-Covid there has been significant focus on grassroots mus

<a id='3-3'></a>
### 3.3 获取相关数据

<a id='ex01'></a>
### 练习 1

在本练习中，你将创建一个函数：输入 `query` 与 `top_k`，输出包含 `relevant items` 的列表。你需要把目前的两个函数整合为一个。

<details>
<summary style="color: green;">提示 1</summary>
用于获取相关索引的函数是 <code>retrieve</code>。请记住它接收 <code>query</code> 和 <code>top_k</code>，并返回<b>索引列表</b>。
</details>
<details>
<summary style="color: green;">提示 2</summary>
用于根据索引获取相关数据的函数是 <code>query_news</code>。它接收<b>一组索引</b>并返回相关数据列表。
</details>

In [ ]:
# 评分单元 

def get_relevant_data(query: str, top_k: int = 5) -> list[dict]:
    """
    根据给定查询检索并返回最相关的数据条目。

    该函数执行以下步骤：
    1. 根据 `query` 从数据集中检索前 k 个最相关条目的索引。
    2. 根据这些索引获取对应的数据内容。

    参数：
    - query (str)：用于查找相关条目的查询字符串。
    - top_k (int, optional)：返回的前 k 条数据，默认值为 5。

    返回：
    - list[dict]：包含最相关条目数据的字典列表。

    """
    ### START CODE HERE ###

    # 根据查询检索 top_k 个最相关条目的索引
    relevant_indices = retrieve(query=query, top_k=top_k)  # @REPLACE EQUALS None

    # 使用上一步索引获取对应条目数据
    relevant_data = query_news(relevant_indices)  # @REPLACE EQUALS None

    ### END CODE HERE ###
    
    return relevant_data

In [9]:
query = "Greatest storms in the US"
relevant_data = get_relevant_data(query, top_k = 1)
pprint(relevant_data)

[
  {
    "guid": "3ca548fe82c3fcae2c4c0c635d03eb2e",
    "title": "Large tornado seen touching down in Nebraska",
    "description": "Severe and powerful storms have moved across several US states, leaving many experiencing power shortages.",
    "venue": "BBC",
    "url": "https://www.bbc.co.uk/news/world-us-canada-68860070",
    "published_at": "2024-04-26",
    "updated_at": "2024-04-28"
  }
]


**期望输出**
```
[{'guid': '3ca548fe82c3fcae2c4c0c635d03eb2e',
  'title': 'Large tornado seen touching down in Nebraska',
  'description': 'Severe and powerful storms have moved across several US '
                 'states, leaving many experiencing power shortages.',
  'venue': 'BBC',
  'url': 'https://www.bbc.co.uk/news/world-us-canada-68860070',
  'published_at': '2024-04-26',
  'updated_at': '2024-04-28'}]
```

In [ ]:
# 运行此单元可对函数进行多项测试。若看到 "All test passed!"，通常说明你的答案也能通过自动评分。
unittests.test_get_relevant_data(get_relevant_data)

 All tests passed!


<a id='3-4'></a>

### 3.4 格式化相关数据


<a id='ex02'></a>

### 练习 2

在本练习中，你将创建 `format_relevant_data` 函数：接收数据列表并将其格式化。

**注意：** 你可以调整排版，但输出中**必须**包含以下信息：

* 新闻标题
* 新闻描述
* 新闻发布日期
* 新闻 URL

你的输出中必须包含关键词 `'title'`、`'url'`、`'published_at'` 和 `'description'`（大小写均可）。这些是评分所需字段。

**提示：** 推荐字符串使用**双引号**，字典键使用**单引号**（例如 `data['title']`）。
下面是一个可参考的格式：

```python
f"""
Title: {news_1_title}, Description: {news_1_description}, Published at: {news_1_published_date}\nURL: {news_1_URL}
Title: {news_2_title}, Description: {news_2_description}, Published at: {news_2_published_date}\nURL: {news_2_URL}
...
Title: {news_k_title}, Description: {news_k_description}, Published at: {news_k_published_date}\nURL: {news_k_URL}
"""
```

<details>  
<summary style="color: green;">提示 1</summary>  
<p>你可以通过字典键访问文档中的各个字段。</p>  
<p>例如获取标题：<code>document['title']</code></p>  
<p>可按如下方式格式化：<code>f"Title: {document['title']}"</code></p>  
</details>

<details>  
<summary style="color: green;">提示 2</summary>  
<p>记得把每条格式化后的文档追加到 <code>formatted_documents</code> 列表：<code>formatted_documents.append(formatted_document)</code>。</p>  
</details>

In [ ]:
# 评分单元

def format_relevant_data(relevant_data):
    """
    将相关文档列表格式化为结构化字符串，供 RAG 系统使用。

    参数：
    relevant_data (list)：相关数据列表。

    返回：
    str：包含相关文档信息的格式化字符串，可用于检索增强生成（RAG）系统。
    """

    ### START CODE HERE ###

    # 创建一个列表用于存储格式化后的文档字符串
    formatted_documents = []
    
    # 遍历每个相关文档
    for document in relevant_data:

        # 将每条文档格式化为结构化字符串。每条文档应单独占一行，因此每条后面要添加换行符。
        formatted_document = f"Title: {document['title']}, Description: {document['description']}, Published at: {document['published_at']}\nURL: {document['url']}" # @REPLACE EQUALS None
        
        # 将格式化后的文档字符串追加到 formatted_documents 列表
        formatted_documents.append(formatted_document) # @REPLACE formatted_documents.append(None)
    
    ### END CODE HERE ###
    
    # 返回最终的增强提示字符串

    return "\n".join(formatted_documents)

In [12]:
example_data = NEWS_DATA[4:8]

现在用一些查询来测试你的函数。

In [13]:
print(format_relevant_data(example_data))

Title: Prosecutors ask for halt to case against Spain PM's wife, Description: Pedro Sánchez is deciding whether to resign after a case against his wife by an anti-corruption group., Published at: 2024-04-25
URL: https://www.bbc.co.uk/news/world-europe-68895727
Title: WATCH: Would you pay a tourist fee to enter Venice?, Description: From Thursday visitors making a trip to the famous city at peak times will be charged a trial entrance fee., Published at: 2024-04-25
URL: https://www.bbc.co.uk/news/world-europe-68898441
Title: Supreme Court divided on whether Trump has immunity, Description: The justices discussed immunity, coups, pardons, Operation Mongoose - and the future of democracy., Published at: 2024-04-25
URL: https://www.bbc.co.uk/news/world-us-canada-68901817
Title: More than 150 killed as heavy rains pound Tanzania, Description: The prime minister warns that El Niño-triggered heavy rains are likely to continue into May., Published at: 2024-04-25
URL: https://www.bbc.co.uk/news/

In [ ]:
# 测试你的函数！
unittests.test_format_relevant_data(format_relevant_data)

 All tests passed!


<a id='3-5'></a>
### 3.5 生成最终提示词

下一个函数已为你提供。它会将 query 与相关数据整合，生成最终提示词。你可以自由修改提示模板并实验不同提示对最终结果的影响！

In [ ]:
# 可编辑单元

def generate_final_prompt(query, top_k=5, use_rag=True, prompt=None):
    """
    根据用户查询生成最终提示词，并可选地通过 RAG 融入相关数据。

    参数：
        query (str)：需要生成提示词的用户查询。
        top_k (int, optional)：检索并融入的相关数据条目数量，默认值为 5。
        use_rag (bool, optional)：是否启用检索增强生成（RAG）
                                  并将相关数据加入提示词，默认值为 True。
        prompt (str, optional)：提示词模板字符串。可包含占位符 {query} 与 {documents}，
                                分别用于插入查询与格式化后的相关数据。

    返回：
        str：生成后的提示词；可能仅包含查询，也可能包含扩展上下文。
    """
    # 若不使用 RAG，则直接返回查询
    if not use_rag:
        return query

    # 根据查询检索前 top_k 条相关数据
    relevant_data = get_relevant_data(query, top_k=top_k)

    # 对检索到的数据进行格式化
    retrieve_data_formatted = format_relevant_data(relevant_data)

    # 若未提供自定义模板，则使用默认模板
    if prompt is None:
        prompt = (
            f"Answer the user query below. There will be provided additional information for you to compose your answer. "
            f"The relevant information provided is from 2024 and it should be added as your overall knowledge to answer the query, "
            f"you should not rely only on this information to answer the query, but add it to your overall knowledge."
            f"Query: {query}\n"
            f"2024 News: {retrieve_data_formatted}"
        )
    else:
        # 若提供了自定义模板，则插入 query 与格式化文档
        prompt = prompt.format(query=query, documents=retrieve_data_formatted)

    return prompt

In [16]:
print(generate_final_prompt("Tell me about the US GDP in the past 3 years."))

Answer the user query below. There will be provided additional information for you to compose your answer. The relevant information provided is from 2024 and it should be added as your overall knowledge to answer the query, you should not rely only on this information to answer the query, but add it to your overall knowledge.Query: Tell me about the US GDP in the past 3 years.
2024 News: Title: America's Economy Is No. 1. That Means Trouble, Description: If you want a single number to capture America’s economic stature, here it is: This year, the U.S. will account for 26.3% of the global gross domestic product, the highest in almost two decades. That’s based on the latest projections from the International Monetary Fund. According to the IMF, Europe’s share of world GDP has dropped 1.4 percentage points since 2018, and Japan’s by 2.1 points. The U.S. share, by contrast, is up 2.3 points., Published at: 2024-04-26
URL: https://www.wsj.com/articles/americas-economy-is-no-1-that-means-tro

<a id='3-6'></a>
### 3.6 LLM 调用

现在把上面的函数整合起来输入 LLM。其参数如下：

- `query`：要传给 LLM 的查询。
- `use_rag`：布尔值，表示是否启用 RAG。该参数可帮助你比较使用与不使用 RAG 时的查询效果。
- `model`：要使用的模型。你可以更换，但标准配置是 Llama 3 Billion 参数模型。

In [ ]:
def llm_call(query, top_k = 5, use_rag = True, prompt = None):
    """
    调用 LLM 基于查询生成回答，并可选启用检索增强生成。

    参数：
        query (str)：将由语言模型处理的用户查询。
        use_rag (bool, optional)：是否启用检索增强生成，
                                  即将相关文档融入提示词。默认值为 True。

    返回：
        str：语言模型生成响应的正文内容。
    """
    

    # 获取包含查询与相关文档的提示词
    prompt = generate_final_prompt(query, top_k, use_rag, prompt)

    # 调用 LLM
    generated_response = generate_with_single_input(prompt)

    # 取出响应正文
    generated_message = generated_response['content']
    
    return generated_message

In [18]:
query = "Tell me about the US GDP in the past 3 years."

In [19]:
print(llm_call(query, use_rag = True))

Based on general economic knowledge up to 2024 and the specific news updates provided for 2024, here is an overview of the US GDP performance over the last few years (2021–2024):

### **2024: Steady Resilience and Global Leadership**
In 2024, the US economy has demonstrated remarkable resilience and re-established its dominance as the world's leading economic power, though the growth pace has moderated compared to the post-pandemic boom.

*   **Global Share:** According to International Monetary Fund (IMF) projections released in mid-2024, the US accounts for **26.3% of global GDP**. This is the highest share in almost two decades, significantly outpacing other major economies. Meanwhile, Europe's share dropped 1.4 percentage points and Japan's fell 2.1 points since 2018, while the US share increased by 2.3 points.
*   **First Quarter Performance:** The economy showed slowing but positive momentum in the first quarter. GDP expanded at a **1.6% seasonally-adjusted annual rate**. While t

In [20]:
print(llm_call(query, use_rag = False))

Over the past three years (2022–2024), the United States economy has demonstrated remarkable resilience, navigating a complex mix of post-pandemic adjustments, high inflation, interest rate hikes, and global volatility. Despite these headwinds, the USDP has grown consistently, driven primarily by domestic consumption rather than business investment or net exports.

Here is a breakdown of the US GDP trajectory over this period:

### 1. Real GDP Growth Rates
Most economic data reported by the Bureau of Economic Analysis (BEA) adjusts for inflation to show "Real GDP" growth, which reflects the true change in economic output.

*   **2022: 2.1% Growth**
    Following the boom years of 2020 and 2021, the US economy cooled in 2022 as the Federal Reserve raised interest rates to combat the highest inflation in four decades. While the growth rate slowed significantly compared to the anomalous 5.7% surge in 2021, it remained well above the Fed's 2% target, avoiding a recession despite the aggres

<a id='4'></a>

## 4 - 体验你的 RAG 系统
---

现在你可以用自己的 query 进行实验，观察系统效果！你可以输入任意 query，系统会分别展示使用 RAG 与不使用 RAG 的回答。请注意，你当前使用的是 2024 年新闻数据集，因此并非所有 query 都适合展示该框架效果。可尝试以下示例问题：

* 过去一年最重要的事件是什么？
* 2024 年全球变暖进展如何？
* 请介绍最近的 AI 进展。
* 请给我过去一年最重要的事实。

你也可以指定增强提示词模板，并用占位符 {query} 与 {documents} 标明其在提示结构中的插入位置。例如：

```
这里是查询：{query}
这里是文档：{documents}
```

In [21]:
display_widget(llm_call)

HTML(value='\n    <style>\n        .custom-output {\n            background-color: #f9f9f9;\n            color…

Text(value='', description='Query:', layout=Layout(width='100%'), placeholder='Type your query here')

Textarea(value='', description='Augmented prompt layout:', layout=Layout(height='100px', width='100%'), placeh…

IntSlider(value=5, description='Top K:', max=20, min=1, style=SliderStyle(description_width='initial'))

Button(description='Get Responses', style=ButtonStyle(button_color='#f0f0f0'))

Output()

恭喜你！你已经构建出第一个简单的 RAG 系统！继续加油！